# Silver Pedido Detalle

**Fuente:** `brz_dataco.sharepoint.raw_supply_chain` (histórico completo acumulado)

**Destino:** Tabla de **Detalle de Pedido**, en `slv_dataco.supply_chain`

**Grano:** un registro por `order_item_id` (línea de detalle de pedido)

## Qué hace

1. Lee todo el histórico de Bronze acumulado, no solo el último corte.
2. Limpia y normaliza los datos:

   * Identificadores: `int`.
   * Cantidades: `int`.
   * Importes monetarios: `decimal(12,2)`.
   * `order_item_discount_rate`: `decimal(5,4)`.
   * `order_item_profit_ratio`: `decimal(9,6)`.
3. Deduplica por `order_item_id`, priorizando el registro del corte más reciente (`dedup_reciente`, `fecha_fin_corte` descendente).
4. Marca la calidad de cada registro (ver más abajo). Las líneas sin `order_item_id` ya no se descartan: se conservan en cuarentena.
5. Conserva `order_id` para relacionar el detalle con la tabla de **Pedidos**.
6. Conserva las columnas de trazabilidad provenientes de Bronze.
7. Escribe la tabla Silver mediante `overwrite` completo.
8. Registra en el esquema de control cuántas filas se agruparon y cuántas quedaron en cuarentena.

## Columnas de negocio

* `order_item_id`
* `order_id`
* `order_item_cardprod_id`
* `order_item_quantity`
* `order_item_product_price`
* `order_item_discount`
* `order_item_discount_rate`
* `order_item_profit_ratio`
* `sales`
* `order_item_total`
* `order_profit_per_order`
* `benefit_per_order`
* `sales_per_customer`

## Calidad y cuarentena

Ningún registro se elimina. Los que incumplen una regla se conservan marcados y Gold no los lee.

| Columna | Valores |
|---|---|
| `estado_calidad` | `VALIDO` o `CUARENTENA` |
| `motivo_cuarentena` | `ITEM_ID_INVALIDO` (`order_item_id` nulo, vacío o 0), `CANTIDAD_INVALIDA` (cantidad menor o igual a 0) o `NULL` (un solo motivo por registro: el primero que se cumpla) |
| `fecha_procesamiento` | Momento en que Silver procesó el registro |

Las líneas sin `order_item_id` no se pueden identificar ni deduplicar, así que se agrupan en un solo registro con `order_item_id` nulo, marcado en cuarentena.

## Trazabilidad conservada

* `fecha_inicio_corte`
* `fecha_fin_corte`
* `fecha_carga`
* `id_ejecucion`

## Auditoría de la ejecución

Al terminar, el notebook deja evidencia en `metadata_dataco.supply_chain`:

* `hd_deduplicacion_log`: una fila por ejecución, con las líneas únicas, las filas agrupadas y los duplicados reales por `order_id` + `order_item_id`, que deben ser **0**.
* `hd_cuarentena_resumen`: filas aisladas por regla en esa ejecución, incluso cuando son 0.

## Nota de diseño

`order_id` se conserva en el detalle para permitir la relación con la tabla de **Pedidos**.

Esta relación es necesaria para los procesos posteriores de Gold que requieren combinar información del pedido con información a nivel de línea de detalle.

La tabla utiliza `dedup_reciente` para priorizar el registro correspondiente al corte más reciente cuando existen múltiples registros para un mismo `order_item_id`.

## Por qué importa este notebook

La tabla de **Detalle de Pedido** mantiene el nivel de granularidad de línea de pedido, sin realizar agregaciones.

Junto con la tabla de **Pedidos**, constituye la base para los indicadores de Gold que requieren información detallada de ventas, costos, descuentos y beneficios. Gold solo lee las líneas con `estado_calidad = 'VALIDO'`.

In [0]:
# ============================================================
# Celda 1: Parámetros + lectura de Bronze + helpers
# ------------------------------------------------------------
# Mismo patrón que silver_pedidos. Sesión propia, se repite todo.
# Los helpers (clean_str, clean_num, tipo_invalido y dedup_reciente)
# vienen del módulo compartido modulos_compartidos.
# ============================================================

import os
import sys

from pyspark.sql import functions as F, Window

# Raíz del repo: dos niveles arriba de la carpeta de este notebook
raiz_repositorio = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if raiz_repositorio not in sys.path:
    sys.path.insert(0, raiz_repositorio)

from modulos_compartidos.transformaciones_silver import (
    clean_str,
    clean_num,
    tipo_invalido,
    dedup_reciente,
)

dbutils.widgets.text("bronze_catalog", "brz_dataco", "Catálogo Bronze")
dbutils.widgets.text("bronze_schema", "sharepoint", "Esquema Bronze")
dbutils.widgets.text("bronze_table", "raw_supply_chain", "Tabla Bronze")

dbutils.widgets.text("silver_catalog", "slv_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "supply_chain", "Esquema Silver")

bronze_catalog = dbutils.widgets.get("bronze_catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
bronze_table = dbutils.widgets.get("bronze_table")

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")

BRONZE_TABLE_FQN = f"{bronze_catalog}.{bronze_schema}.{bronze_table}"
SILVER_PEDIDO_DETALLE_FQN = f"{silver_catalog}.{silver_schema}.hd_pedido_detalle"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{silver_catalog}.{silver_schema}"
)

df_bronze = spark.table(BRONZE_TABLE_FQN)

print(f"Total filas en Bronze: {df_bronze.count()}")
print(f"✓ Helpers importados desde modulos_compartidos (raíz: {raiz_repositorio})")

In [0]:
# ============================================================
# INICIO DE LOG DE EJECUCIÓN — slv_order_details
# ============================================================

import uuid

dbutils.widgets.text("metadata_catalog", "metadata_dataco", "Catálogo Metadata")
dbutils.widgets.text("metadata_schema", "supply_chain", "Esquema Metadata")
dbutils.widgets.text("id_ejecucion_manual", "", "Id de ejecución (solo corridas manuales)")

metadata_catalog = dbutils.widgets.get("metadata_catalog")
metadata_schema = dbutils.widgets.get("metadata_schema")

TABLA_ETL_LOG = f"{metadata_catalog}.{metadata_schema}.hd_etl_log"
TABLA_DEDUP_LOG = f"{metadata_catalog}.{metadata_schema}.hd_deduplicacion_log"
TABLA_CUARENTENA_RESUMEN = f"{metadata_catalog}.{metadata_schema}.hd_cuarentena_resumen"

id_manual = dbutils.widgets.get("id_ejecucion_manual")
try:
    id_ejecucion = dbutils.jobs.taskValues.get(taskKey="prepare_pipeline_control", key="id_ejecucion")
except Exception:
    if not id_manual:
        raise
    id_ejecucion = id_manual

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "slv_order_details"
CAPA = "silver"

spark.sql(f"""
INSERT INTO {TABLA_ETL_LOG}
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

try:
    dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)
    print("✓ log_id publicado como Task Value")
except Exception:
    if not id_manual:
        raise
    print("• Corrida manual: log_id no se publica como Task Value")

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
[c for c in df_bronze.columns if "order_item" in c.lower() or c.lower() in ("sales", "order_profit_per_order", "benefit_per_order", "sales_per_customer")]

In [0]:
# ============================================================
# Celda 2: Limpieza + deduplicación + cuarentena de hd_pedido_detalle
# ------------------------------------------------------------
# 1) Limpia y tipifica desde Bronze (sin deduplicar aún).
# 2) Deduplica por order_item_id, priorizando el corte más reciente
#    (fecha_fin_corte descendente).
# 3) Ningún registro se elimina: order_item_id nulo, vacío o 0 ->
#    ITEM_ID_INVALIDO; texto no numérico en una columna numérica ->
#    TIPO_DATO_INVALIDO; cantidad vacía o <= 0 -> CANTIDAD_INVALIDA.
#    Las líneas sin ID se agrupan en un solo registro (order_item_id nulo).
# 4) Se conservan las columnas de trazabilidad provenientes de Bronze.
# ============================================================

from functools import reduce

COLUMNAS_NUMERICAS = {
    "order_id": "int", "order_item_cardprod_id": "int", "order_item_quantity": "int",
    "order_item_product_price": "decimal(12,2)", "order_item_discount": "decimal(12,2)",
    "order_item_discount_rate": "decimal(5,4)", "order_item_profit_ratio": "decimal(9,6)",
    "sales": "decimal(12,2)", "order_item_total": "decimal(12,2)",
    "order_profit_per_order": "decimal(12,2)", "benefit_per_order": "decimal(12,2)",
    "sales_per_customer": "decimal(12,2)",
}
tipo_dato_invalido = reduce(lambda a, b: a | b,
                            [tipo_invalido(c, t) for c, t in COLUMNAS_NUMERICAS.items()])

df_detalle_limpio = df_bronze.select(
    clean_num("order_item_id", "int").alias("order_item_id"),
    clean_num("order_id", "int").alias("order_id"),
    clean_num("order_item_cardprod_id", "int").alias("order_item_cardprod_id"),
    clean_num("order_item_quantity", "int").alias("order_item_quantity"),
    clean_num("order_item_product_price", "decimal(12,2)").alias("order_item_product_price"),
    clean_num("order_item_discount", "decimal(12,2)").alias("order_item_discount"),
    clean_num("order_item_discount_rate", "decimal(5,4)").alias("order_item_discount_rate"),
    clean_num("order_item_profit_ratio", "decimal(9,6)").alias("order_item_profit_ratio"),
    clean_num("sales", "decimal(12,2)").alias("sales"),
    clean_num("order_item_total", "decimal(12,2)").alias("order_item_total"),
    clean_num("order_profit_per_order", "decimal(12,2)").alias("order_profit_per_order"),
    clean_num("benefit_per_order", "decimal(12,2)").alias("benefit_per_order"),
    clean_num("sales_per_customer", "decimal(12,2)").alias("sales_per_customer"),

    # Calidad: texto no vacío que no se puede convertir a número
    tipo_dato_invalido.alias("tipo_dato_invalido"),

    # --------------------------------------------------------
    # Trazabilidad
    # --------------------------------------------------------
    F.col("fecha_inicio_corte"),
    F.col("fecha_fin_corte"),
    F.col("fecha_carga"),
    F.col("id_ejecucion")
)

df_silver_pedido_detalle = (
    dedup_reciente(df_detalle_limpio, ["order_item_id"], orden_col="fecha_fin_corte")
    .withColumn("motivo_cuarentena",
                F.when(F.col("order_item_id").isNull() | (F.col("order_item_id") == 0), F.lit("ITEM_ID_INVALIDO"))
                 .when(F.col("tipo_dato_invalido"), F.lit("TIPO_DATO_INVALIDO"))
                 .when(F.col("order_item_quantity").isNull() | (F.col("order_item_quantity") <= 0), F.lit("CANTIDAD_INVALIDA")))
    .withColumn("estado_calidad",
                F.when(F.col("motivo_cuarentena").isNotNull(), F.lit("CUARENTENA"))
                 .otherwise(F.lit("VALIDO")))
    .withColumn("fecha_procesamiento", F.current_timestamp())
    .drop("tipo_dato_invalido")
)

print(f"Filas en Bronze                          : {df_bronze.count()}")
print(f"Filas en Silver (después del dedup)      : {df_silver_pedido_detalle.count()}")

display(df_silver_pedido_detalle.limit(10))

In [0]:
# ============================================================
# Celda 3: Escritura a silver_pedido_detalle
# ------------------------------------------------------------
# overwrite completo, mismo patrón que el resto de Silver.
# ============================================================

(
    df_silver_pedido_detalle.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_PEDIDO_DETALLE_FQN)
)

print(f"✓ {df_silver_pedido_detalle.count()} filas escritas en {SILVER_PEDIDO_DETALLE_FQN}")

In [0]:
# ============================================================
# Registro de deduplicación y cuarentena de esta ejecución
# ------------------------------------------------------------
# filas_agrupadas   -> informativa: filas de Bronze que no quedan como línea única.
# duplicados_reales -> filas repetidas de (order_id, order_item_id) en Bronze; debe ser 0.
# ============================================================
TABLA_LOG = "hd_pedido_detalle"
MOTIVOS = ["ITEM_ID_INVALIDO", "CANTIDAD_INVALIDA", "TIPO_DATO_INVALIDO"]

filas_entrada = df_bronze.count()
filas_salida = df_silver_pedido_detalle.count()
filas_unicas_bronze = df_bronze.select("order_id", "order_item_id").distinct().count()
duplicados_reales = filas_entrada - filas_unicas_bronze

spark.sql(
    f"DELETE FROM {TABLA_DEDUP_LOG} WHERE id_ejecucion = :id AND tabla_silver = :tabla",
    args={"id": id_ejecucion, "tabla": TABLA_LOG}
)
spark.sql(
    f"INSERT INTO {TABLA_DEDUP_LOG} VALUES (:id, :tabla, :clave, :entrada, :salida, "
    f":agrupadas, :clave_fisica, :duplicados, current_timestamp())",
    args={"id": id_ejecucion, "tabla": TABLA_LOG, "clave": "order_item_id",
          "entrada": filas_entrada, "salida": filas_salida,
          "agrupadas": filas_entrada - filas_salida,
          "clave_fisica": "order_id, order_item_id",
          "duplicados": duplicados_reales}
)

spark.sql(
    f"DELETE FROM {TABLA_CUARENTENA_RESUMEN} WHERE id_ejecucion = :id AND tabla_silver = :tabla",
    args={"id": id_ejecucion, "tabla": TABLA_LOG}
)
df_motivos = spark.createDataFrame([(m,) for m in MOTIVOS], "motivo STRING")
df_conteo = (df_silver_pedido_detalle.filter("estado_calidad = 'CUARENTENA'")
             .groupBy("motivo_cuarentena").count())
(
    df_motivos.join(df_conteo, df_motivos.motivo == df_conteo.motivo_cuarentena, "left")
    .select(
        F.lit(id_ejecucion).alias("id_ejecucion"),
        F.lit(TABLA_LOG).alias("tabla_silver"),
        F.col("motivo").alias("motivo_cuarentena"),
        F.coalesce(F.col("count"), F.lit(0)).cast("bigint").alias("filas"),
        F.current_timestamp().alias("fecha_registro"),
    )
    .write.format("delta").mode("append").saveAsTable(TABLA_CUARENTENA_RESUMEN)
)

print(f"✓ {filas_entrada} filas de Bronze -> {filas_salida} líneas | agrupadas: {filas_entrada - filas_salida} | duplicados reales: {duplicados_reales}")

In [0]:
TABLA_DESTINO = SILVER_PEDIDO_DETALLE_FQN   # cambia por notebook (ver tabla)
filas_procesadas = spark.table(TABLA_DESTINO).count()

spark.sql(f"""
UPDATE {TABLA_ETL_LOG}
SET estado = 'EXITOSO',
    fecha_fin = current_timestamp(),
    filas_procesadas = {filas_procesadas}
WHERE log_id = '{LOG_ID}'
""")

print(f"✓ etl_log actualizado: EXITOSO ({filas_procesadas} filas en {TABLA_DESTINO})")